In [1]:
#%pip install mlflow==2.7.0 scikit-learn pandas
import mlflow, json, os, pandas as pd
from sklearn.metrics import classification_report, accuracy_score


In [2]:
ROOT = os.getcwd()
RESP_CSV = os.path.join(ROOT, "G:\\Resume-Matcher\\experiments\\prompts\\results\\responses.csv")            # your responses file
HUMAN_CSV = os.path.join(ROOT, "G:\\Resume-Matcher\\experiments\\prompts\\results\\human_rubric_template.csv")      # human rubric file produced earlier
EVAL_CSV = os.path.join(ROOT, "G:\\Resume-Matcher\\experiments\\prompts\\results\\evaluation_results.csv")           # quantitative metrics CSV you created
REPORT_MD = os.path.join(ROOT, "experiments/prompts/prompt_report.md")


In [ ]:
# load responses (long format with columns: id, strategy, assigned_label, ground_truth)
df = pd.read_csv(RESP_CSV)
# convert wide -> long
rows = []
for _, r in df.iterrows():
    rows.append({"id": r["id"], "strategy":"zero", "response_text": r["zero_text"], "assigned_label": r["zero_label"], "ground_truth": r["ground_truth"]})
    rows.append({"id": r["id"], "strategy":"few",  "response_text": r["few_text"],  "assigned_label": r["few_label"],  "ground_truth": r["ground_truth"]})
    rows.append({"id": r["id"], "strategy":"adv",  "response_text": r["adv_text"],  "assigned_label": r["adv_label"],  "ground_truth": r["ground_truth"]})
long = pd.DataFrame(rows)

# normalize labels to High/Medium/Low
def norm(x):
    if pd.isna(x): return x
    s = str(x).lower()
    if "high" in s: return "High"
    if "medium" in s: return "Medium"
    if "low" in s: return "Low"
    return x
long["assigned_label"] = long["assigned_label"].apply(norm)
long["ground_truth"] = long["ground_truth"].apply(norm)

# compute per-strategy accuracy + classification report
summary = {}
for strat, g in long.groupby("strategy"):
    acc = (g["assigned_label"] == g["ground_truth"]).mean()
    report = classification_report(g["ground_truth"], g["assigned_label"], zero_division=0, output_dict=True)
    summary[strat] = {"accuracy": float(acc), "counts": g["assigned_label"].value_counts().to_dict(), "report": report}

# save summary
OUT_SUM = "G:/Resume-Matcher/experiments/prompts/quant_summary.json"

os.makedirs(os.path.dirname(OUT_SUM) or ".", exist_ok=True)
with open(OUT_SUM, "w") as f:
    json.dump(summary, f, indent=2)

# optional: write the long table so human rubric is ready
long.to_csv("human_rubric_filled_from_responses.csv", index=False)

print("Done — summary saved to", OUT_SUM)


Done — summary saved to G:/Resume-Matcher/experiments/prompts/quant_summary.json


In [13]:
with open(OUT_SUM, "w") as f:
    json.dump(report, f, indent=2)

mlflow.set_experiment("prompt_eval")
with mlflow.start_run(run_name="prompt_strategies_local"):
    # log per-strategy metrics
    for strat, stats in summary.items():
        mlflow.log_metric(f"{strat}_accuracy", stats["accuracy"])
        # log counts as metrics
        for label,count in stats["counts"].items():
            mlflow.log_metric(f"{strat}_count_{label}", int(count))
    # log artifacts (responses, human rubric, quant summary)
    mlflow.log_artifact(RESP_CSV, artifact_path="responses")
    mlflow.log_artifact(HUMAN_CSV, artifact_path="human_rubric")
    mlflow.log_artifact(os.path.join("G:\\Resume-Matcher\\experiments\\prompts\\quant_summary.json"), artifact_path="summary")
    # optional: save classification reports per strategy
    for strat, stats in summary.items():
        rep_path = f"G:/Resume-Matcher/experiments/prompts/{strat}_classif.json"
        with open(rep_path,"w") as f: json.dump(stats["classification_report"], f, indent=2)
        mlflow.log_artifact(rep_path, artifact_path=f"classif_reports/{strat}")
print("MLflow logging done. Run `mlflow ui` and open http://localhost:5000 to view.")


MLflow logging done. Run `mlflow ui` and open http://localhost:5000 to view.


In [14]:
import mlflow
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiments:", mlflow.search_experiments())
for e in mlflow.search_experiments():
    print("Experiment:", e.name, e.experiment_id)
    runs = mlflow.search_runs(experiment_ids=[e.experiment_id])
    print("Runs for", e.name, "->", len(runs))
    display(runs.head())


Tracking URI: file:///g:/Resume-Matcher/experiments/prompts/mlruns
Experiments: [<Experiment: artifact_location='file:///g:/Resume-Matcher/experiments/prompts/mlruns/140077499188726403', creation_time=1764363765736, experiment_id='140077499188726403', last_update_time=1764363765736, lifecycle_stage='active', name='prompt_eval', tags={}>, <Experiment: artifact_location='file:///g:/Resume-Matcher/experiments/prompts/mlruns/0', creation_time=1764363765659, experiment_id='0', last_update_time=1764363765659, lifecycle_stage='active', name='Default', tags={}>]
Experiment: prompt_eval 140077499188726403
Runs for prompt_eval -> 5


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.adv_count_Medium,metrics.few_count_Low,metrics.zero_count_High,metrics.zero_accuracy,...,metrics.zero_count_Medium,metrics.few_count_Medium,metrics.few_count_High,metrics.Zero-shot_accuracy,metrics.Few-shot_accuracy,metrics.CoT/Meta_accuracy,tags.mlflow.source.name,tags.mlflow.source.type,tags.mlflow.user,tags.mlflow.runName
0,3867518a43d944dd965041fa502547e6,140077499188726403,FINISHED,file:///g:/Resume-Matcher/experiments/prompts/...,2025-11-29 12:36:54.458000+00:00,2025-11-29 12:36:54.578000+00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,C:\Users\SAHIL\AppData\Roaming\Python\Python31...,LOCAL,SAHIL,prompt_strategies_local
1,57ac3c1a96a344b9a4441365f537b28c,140077499188726403,FAILED,file:///g:/Resume-Matcher/experiments/prompts/...,2025-11-29 12:19:11.943000+00:00,2025-11-29 12:19:12.310000+00:00,5.0,6.0,6.0,0.6,...,14.0,2.0,12.0,NaN,NaN,NaN,C:\Users\SAHIL\AppData\Roaming\Python\Python31...,LOCAL,SAHIL,prompt_strategies_local
2,848019b94ad345469896712d769723a4,140077499188726403,FAILED,file:///g:/Resume-Matcher/experiments/prompts/...,2025-11-29 12:13:29.751000+00:00,2025-11-29 12:13:30.258000+00:00,5.0,6.0,6.0,0.6,...,14.0,2.0,12.0,NaN,NaN,NaN,C:\Users\SAHIL\AppData\Roaming\Python\Python31...,LOCAL,SAHIL,prompt_strategies_local
3,2498528804634f7ba6fa99284d54c659,140077499188726403,FAILED,file:///g:/Resume-Matcher/experiments/prompts/...,2025-11-29 12:11:06.544000+00:00,2025-11-29 12:11:07.039000+00:00,5.0,6.0,6.0,0.6,...,14.0,2.0,12.0,NaN,NaN,NaN,C:\Users\SAHIL\AppData\Roaming\Python\Python31...,LOCAL,SAHIL,prompt_strategies_local
4,55dd43fa6e0645cba2d05bd350747f15,140077499188726403,FINISHED,file:///g:/Resume-Matcher/experiments/prompts/...,2025-11-28 21:02:46.133000+00:00,2025-11-28 21:02:46.530000+00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.6,0.75,0.55,C:\Users\SAHIL\AppData\Roaming\Python\Python31...,LOCAL,SAHIL,prompt_strategies


Experiment: Default 0
Runs for Default -> 0


,run_id,experiment_id,status,artifact_uri,start_time,end_time
